In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import time
import sys
import pickle
import tensorflow as tf
import keras
import hls4ml
from sklearn.metrics import accuracy_score
from keras.models import Model
from pathlib import Path
from tensorflow.keras import optimizers
from hls4ml.model.profiling import numerical
from cnn_base import build_qkeras_model


TRAIN_PICKLE_PATH = "./dataset/sat6_train.pkl"
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"
DATA_DIR = "./dataset/sat-6-full.mat"
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 3e-4
 
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

def trace_layers(config):
    config['Model']['TraceOutput'] = True
    for layer in config['LayerName']:
        config['LayerName'][layer]['Trace'] = True


def print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key), end='')
        if isinstance(value, dict):
            print()
            print_dict(value, indent + 1)
        else:
            print(':' + ' ' * (20 - len(key) - 2 * indent) + str(value))
 
def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]


2026-09-14 08:09:18.614420: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-14 08:09:18.616399: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-14 08:09:18.635187: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-14 08:09:18.635686: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-14 08:09:19.073185: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

In [ ]:
X_train_all, y_train_all = load_pickle_data(TRAIN_PICKLE_PATH)
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)

X_train_all = X_train_all.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

val_idx = int(0.75 * len(X_train_all))
X_train, y_train = X_train_all[:val_idx], y_train_all[:val_idx]
X_val, y_val = X_train_all[val_idx:], y_train_all[val_idx:]

n_classes = 6
input_shape = X_train.shape[1:]
train_size = len(X_train)
input_shape, train_size

Lendo ./dataset/sat6_train.pkl ...
Lendo ./dataset/sat6_test.pkl ...


((32, 32, 4), 243000)

In [3]:
filters_per_conv_layer = [20, 8]
#neurons_per_dense_layer = [42]

model = build_qkeras_model(input_shape,n_classes, filters_per_conv_layer)
model.summary()

Model: "qkeras_cnn"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 32, 4)]       0         
                                                                 
 conv_0 (QConv2D)            (None, 30, 30, 20)        720       
                                                                 
 conv_act_0 (QActivation)    (None, 30, 30, 20)        0         
                                                                 
 conv_1 (QConv2D)            (None, 28, 28, 8)         1440      
                                                                 
 conv_act_1 (QActivation)    (None, 28, 28, 8)         0         
                                                                 
 flatten (Flatten)           (None, 6272)              0         
                                                                 
 output_dense (Dense)        (None, 6)                 3

In [ ]:
model.compile(
#    optimizer=keras.optimizers.Adam(learning_rate=5e-3, beta_1=0.9, beta_2=0.999, epsilon=1e-7, amsgrad=True),
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy'],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]
t0 = time.time()
model.fit(X_train, y_train, epochs=20, validation_data=(X_val, y_val), callbacks=callbacks)
print(f'\nTraining took {(time.time() - t0) / 60:.1f} minutes')
os.makedirs('./models', exist_ok=True)
model.save('./models/qkeras_cnn_part4a.keras')

In [ ]:
hls_config = hls4ml.utils.config_from_keras_model(
    model, granularity='name', backend='Vitis', 
)
hls_config['LayerName']['output_dense']['Precision']['result'] = 'fixed<16,6,RND,SAT>'

trace_layers(hls_config)
print("HLS4ML configuration:")
print_dict(hls_config)

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=hls_config,
    backend='Vitis',
    output_dir='../hls4ml_prjs/hls4ml_prj_cnn_part4a',
    part="xc7z020clg400-1",
    io_type='io_stream',
)
hls_model.compile()

HLS4ML configuration:
Model
  Precision
    default:         fixed<16,6>
  ReuseFactor:       1
  Strategy:          Latency
  BramFactor:        1000000000
  TraceOutput:       True
LayerName
  conv1_input
    Trace:           True
    Precision
      result:        auto
  conv1
    Trace:           True
    Precision
      result:        auto
      weight:        fixed<8,1,TRN,WRAP,0>
      bias:          auto
      accum:         auto
    ReuseFactor:     1
    ParallelizationFactor:1
    ConvImplementation:LineBuffer
  conv1_linear
    Trace:           True
    Precision
      result:        auto
      table:         fixed<18,8,TRN,WRAP,0>
    ReuseFactor:     1
    TableSize:       1024
  relu1
    Trace:           True
    Precision
      result:        ufixed<8,0,RND_CONV,SAT,0>
      table:         fixed<18,8,TRN,WRAP,0>
    ReuseFactor:     1
    TableSize:       1024
  conv2
    Trace:           True
    Precision
      result:        auto
      weight:        fixed<8,1,TRN,W

In [ ]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

In [ ]:
numerical(model=model, hls_model=hls_model, X=X_test[:2000], plot='boxplot')

In [ ]:
X_test_reduced = np.ascontiguousarray(X_test[:3000])
Y_test_reduced = np.array(y_test[:3000])

y_pred_reduced = model.predict(X_test_reduced)
y_pred_hls = hls_model.predict(X_test_reduced)

print(f'QKeras accuracy : {accuracy_score(np.argmax(Y_test_reduced, axis=1), np.argmax(y_pred_reduced, axis=1)):.4f}')
print(f'hls4ml accuracy : {accuracy_score(np.argmax(Y_test_reduced, axis=1), np.argmax(y_pred_hls,     axis=1)):.4f}')

94/94 [==============================] - 0s 851us/step
QKeras accuracy : 0.9817
hls4ml accuracy : 0.2753


In [ ]:
hls_model.build(csim=False, synth=True, vsynth=True)